# Political Speech Analysis - LLM Models Comparison

In this notebook, we compare the performance of four different BERT models for sentiment and political ideology analysis in political speeches. "BERT is a model for natural language processing developed by Google that learns bi-directional representations of text to significantly improve contextual understanding of unlabeled text across many different tasks." ([Source](https://www.nvidia.com/en-us/glossary/bert/))

---

### **Model 1 - DistilBERT**
DistilBERT is a lighter version of the popular BERT model. It provides an efficient balance between accuracy and computational resources. For sentiment analysis, DistilBERT has been fine-tuned on various text-based sentiment data and is known to perform well on a wide range of tasks.
- **Documentation**: [DistilBERT on Hugging Face](https://huggingface.co/distilbert-base-uncased)

---

### **Model 2 - RoBERTa (cardiffnlp/twitter-roberta-base-sentiment)**
RoBERTa is used for sentiment analysis specifically for sentiment in tweets, which aligns well with political speeches where brevity and emotive content are important.
- **Documentation**: [cardiffnlp/twitter-roberta-base-sentiment on Hugging Face](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment)

---

### **Model 3 - EleutherAI/gpt-neo-2.7B**
The GPT-Neo 2.7B is a large open-source variant of GPT designed for generating human-like text. It was trained to excel in text-based generation and understanding, which also makes it highly effective in analyzing the political content of speech. In this notebook it will be used for ideology prediction (identifying political alignments).
- **Documentation**: [EleutherAI GPT-Neo-2.7B on Hugging Face](https://huggingface.co/EleutherAI/gpt-neo-2.7B)

---

### **Model 4 - RuBERT-Tweet Sentiment (blanchefort/rubert-tweet-sentiment-3)**
RuBERT is a variation of BERT, adapted for the Russian language and fine-tuned to analyze sentiment from short pieces of text, such as tweets. 
- **Documentation**: [blanchefort/rubert-tweet-sentiment-3 on Hugging Face](https://huggingface.co/blanchefort/rubert-base-cased-sentiment-mokoron/blame/ac1072fa3bd07a4c2f536c1bfd87aa6144a1ce46/README.md)

### DistilBERT

In [1]:
import re
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
import torch
import pandas as pd
import logging

# Disable SSL warnings
import requests
requests.packages.urllib3.disable_warnings(requests.packages.urllib3.exceptions.InsecureRequestWarning)

# Set environment variables
torch.set_num_threads(1)  # Set to 1 thread for optimal performance
logging.getLogger("transformers").setLevel(logging.ERROR)

# Model paths for sentiment and ideology analysis
sentiment_model_path = "C:/Users/Admin/Downloads/Machine-Learning/Project_part1/Models/distilbert-base-uncased-finetuned-sst-2-english"
ideology_model_name = "EleutherAI/gpt-neo-2.7B"  # Placeholder for ideology model

# Utility function to load models (local)
def load_model(model_path):
    try:
        model = AutoModelForSequenceClassification.from_pretrained(model_path)
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        return model, tokenizer
    except Exception as e:
        logging.error(f"Error loading model from {model_path}: {str(e)}")
        raise e

# Utility function to chunk large texts
def chunk_text(text, max_length=512):
    words = text.split()
    chunks = [' '.join(words[i:i + max_length]) for i in range(0, len(words), max_length)]
    return chunks

# Utility to analyze sentiment with correct labels
def analyze_sentiment(text, model, tokenizer):
    sentiment_analyzer = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)
    chunks = chunk_text(text)

    sentiments = []
    for chunk in chunks:
        sentiment_result = sentiment_analyzer(chunk)
        sentiment_label = sentiment_result[0]["label"]
        sentiment_score = sentiment_result[0]["score"]

        # Convert sentiment labels from model format to correct labels
        if sentiment_label == "LABEL_0":
            sentiment_label = "NEGATIVE"
        elif sentiment_label == "LABEL_1":
            sentiment_label = "NEUTRAL"
        elif sentiment_label == "LABEL_2":
            sentiment_label = "POSITIVE"
        
        sentiments.append((sentiment_label, sentiment_score))
    
    avg_score = sum([score for _, score in sentiments]) / len(sentiments)
    avg_label = sentiments[0][0]  # Assumes the sentiment doesn't change across chunks
    return avg_label, avg_score

# Utility to analyze ideology (for example, GPT-Neo)
def analyze_ideology(text, model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForSequenceClassification.from_pretrained(model_name)

    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    input_ids = inputs["input_ids"].long()

    outputs = model(input_ids=input_ids)
    predictions = torch.argmax(outputs.logits, dim=1)

    ideology_labels = ["Progressive", "Conservative", "Neutral"]
    return ideology_labels[predictions.item()]

# Split the transcript into two segments
def split_transcript_by_speaker(transcript):
    speaker_1_speech = re.findall(r"\[Speaker 1\]([^\[]+)", transcript)
    speaker_2_speech = re.findall(r"\[Speaker 2\]([^\[]+)", transcript)
    speaker_3_speech = re.findall(r"\[Speaker 3\]([^\[]+)", transcript)

    speaker_1_text = " ".join(speaker_1_speech).strip()
    speaker_2_text = " ".join(speaker_2_speech).strip()
    speaker_3_text = " ".join(speaker_3_speech).strip()
    
    return speaker_1_text, speaker_2_text, speaker_3_text

# Transcript
transcript = """
[Speaker 1]  Kamala Harris. What's up, good to be here. You have fun. Thank you.
[Speaker 2]  We have inflation like very few people have ever seen before, probably the worst in our nation's history. This has been a disaster for people, for the middle class, but for every class.
[Speaker 3]  Such as today we are apple arriaple minutes. człowiek grasp giving the separately dr K mark.
[Speaker 3]  Welcome to you both. It's wonderful to have you. It's an honor to have you both here tonight.
[Speaker 2]  She's a Marxist. Everybody knows she's a Marxist. Her father is a Marxist professor in economics, and he taught her well. But her vice presidential pick says abortion in the ninth month is absolutely fine. He also says execution after birth. It's execution no longer abortion because the baby is born is OK. And that's not OK with me.
[Speaker 1]  One does not have to abandon their faith or deeply held beliefs to agree. The government and Donald Trump certainly should not be telling a woman what to do with her body. Pregnant women who want to carry a pregnancy to term suffering from a miscarriage being denied care in an emergency room because the healthcare providers are afraid they might go to jail and she's bleeding out in a car in the parking lot. She didn't want that. Her husband didn't want that.
[Speaker 2]  In Springfield, they're eating the dogs, the people that came in, they're eating the cats, they're eating the pets of the people that live there. And this is what's happening in our country.
[Speaker 3]  The city manager says there's no evidence of that vice president. I'll let you respond to the rest of what you've heard.
[Speaker 2]  Select Einsparter Assence A
[Speaker 1]  It's my day. Alright Alexa, come on there. It is my day. Life.
[Speaker 1]  You talk about extreme.
[Speaker 3]  Are you now acknowledging that you've lost in 2020? No, I don't acknowledge it at all. I said that sarcastically, you know that.
[Speaker 2]  Now. I don't acknowledge it at all. I said that sarcastically. You know that. It was said, oh, we lost by a whisker. That was said sarcastically.
[Speaker 3]  Mr. President, thank you. Vice President Harris, you heard the President there tonight. He said he didn't say that, that he lost by Whiskers. So he still believes he did not lose the election. That was won by President Biden and yourself.
[Speaker 1]  Donald Trump was fired by 81 million people. So let's be clear about that. And clearly he is having a very difficult time processing that. World leaders are laughing at Donald Trump. I have talked with military leaders, some of whom work with you. And they say you're a disgrace. Understand what it would mean if Donald Trump were back in the White House with no guardrails. Because certainly we know now the court won't stop him. We know JD Vance is not going to stop him. It's up to the American people.
[Speaker 2]  This is the one that weaponized, not me. She weaponized. I probably took a bullet to their head because of the things that they say about me. They talk about democracy. I'm a threat to democracy. They're the threat to democracy.
[Speaker 1]  So I think you've heard tonight two very different visions for our country, one that is focused on the future, and the other that is focused on the past, and an attempt to take us backward.
[Speaker 1]  But we're not going back.
[Speaker 2]  They've had three and a half years to create jobs and all the things we talked about. Why hasn't she done it? The worst president, the worst vice president in the history of our country.
"""

# Perform analysis on each speaker's text
speaker_1_text, speaker_2_text, speaker_3_text = split_transcript_by_speaker(transcript)

# Print each speaker's text
print("Speaker 1 Text:")
print(speaker_1_text)
print("\nSpeaker 2 Text:")
print(speaker_2_text)
print("\nSpeaker 3 Text:")
print(speaker_3_text)

# Load the sentiment model
sentiment_model, sentiment_tokenizer = load_model(sentiment_model_path)

# Analyze Speaker 1
print("\nAnalyzing Speaker 1 using DistilBERT...")
sentiment_label_1, sentiment_score_1 = analyze_sentiment(speaker_1_text, sentiment_model, sentiment_tokenizer)
ideology_label_1 = analyze_ideology(speaker_1_text, ideology_model_name)

# Analyze Speaker 2
print("\nAnalyzing Speaker 2 using DistilBERT...")
sentiment_label_2, sentiment_score_2 = analyze_sentiment(speaker_2_text, sentiment_model, sentiment_tokenizer)
ideology_label_2 = analyze_ideology(speaker_2_text, ideology_model_name)

# Analyze Speaker 3
print("\nAnalyzing Speaker 3 using DistilBERT...")
sentiment_label_3, sentiment_score_3 = analyze_sentiment(speaker_3_text, sentiment_model, sentiment_tokenizer)
ideology_label_3 = analyze_ideology(speaker_3_text, ideology_model_name)

# Store the results
results = {
    "Model": "DistilBERT",
    "Speaker 1 Sentiment": (sentiment_label_1, sentiment_score_1),
    "Speaker 1 Ideology": ideology_label_1,
    "Speaker 2 Sentiment": (sentiment_label_2, sentiment_score_2),
    "Speaker 2 Ideology": ideology_label_2,
    "Speaker 3 Sentiment": (sentiment_label_3, sentiment_score_3),
    "Speaker 3 Ideology": ideology_label_3
}

# Display the results
df_results = pd.DataFrame([results])
print("\nAnalysis Results:")
print(df_results)


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Speaker 1 Text:
Kamala Harris. What's up, good to be here. You have fun. Thank you.
   One does not have to abandon their faith or deeply held beliefs to agree. The government and Donald Trump certainly should not be telling a woman what to do with her body. Pregnant women who want to carry a pregnancy to term suffering from a miscarriage being denied care in an emergency room because the healthcare providers are afraid they might go to jail and she's bleeding out in a car in the parking lot. She didn't want that. Her husband didn't want that.
   It's my day. Alright Alexa, come on there. It is my day. Life.
   You talk about extreme.
   Donald Trump was fired by 81 million people. So let's be clear about that. And clearly he is having a very difficult time processing that. World leaders are laughing at Donald Trump. I have talked with military leaders, some of whom work with you. And they say you're a disgrace. Understand what it would mean if Donald Trump were back in the White House

## RoBERTa

In [ ]:
import re
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
import torch
import pandas as pd
import logging

# Disable SSL warnings
import requests
requests.packages.urllib3.disable_warnings(requests.packages.urllib3.exceptions.InsecureRequestWarning)

# Set environment variables
torch.set_num_threads(1)  # Set to 1 thread for optimal performance
logging.getLogger("transformers").setLevel(logging.ERROR)

# Model paths for sentiment and ideology analysis
sentiment_model_path = "C:/Users/Admin/Downloads/Machine-Learning/Project_part1/Models/robertas"
ideology_model_name = "EleutherAI/gpt-neo-2.7B"  # Placeholder for ideology model

# Utility function to load models (local)
def load_model(model_path):
    try:
        model = AutoModelForSequenceClassification.from_pretrained(model_path)
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        return model, tokenizer
    except Exception as e:
        logging.error(f"Error loading model from {model_path}: {str(e)}")
        raise e

# Utility function to chunk large texts
def chunk_text(text, max_length=512):
    words = text.split()
    chunks = [' '.join(words[i:i+max_length]) for i in range(0, len(words), max_length)]
    return chunks

# Utility to analyze sentiment
def analyze_sentiment(text, model, tokenizer):
    sentiment_analyzer = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)
    chunks = chunk_text(text)
    
    sentiments = []
    for chunk in chunks:
        sentiment_result = sentiment_analyzer(chunk)
        sentiment_label = sentiment_result[0]["label"]
        sentiment_score = sentiment_result[0]["score"]
        sentiments.append((sentiment_label, sentiment_score))
    
    avg_score = sum([score for _, score in sentiments]) / len(sentiments)
    avg_label = sentiments[0][0]  # Assuming the first label is representative
    return avg_label, avg_score

# Utility to analyze ideology (for example, GPT-Neo)
def analyze_ideology(text, model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForSequenceClassification.from_pretrained(model_name)

    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    input_ids = inputs["input_ids"].long()

    outputs = model(input_ids=input_ids)
    predictions = torch.argmax(outputs.logits, dim=1)

    # Custom ideology labels - adjust this as per your fine-tuned model's actual output
    ideology_labels = ["Progressive", "Conservative", "Neutral"]
    return ideology_labels[predictions.item()]

# Transcript
transcript = """
[Speaker 1]  Kamala Harris. What's up, good to be here. You have fun. Thank you.
[Speaker 2]  to
[Speaker 3]  Such as today we are apple arriaple minutes. człowiek grasp giving the separately dr K mark.
[Speaker 3]  Welcome to you both. It's wonderful to have you. It's an honor to have you both here tonight.
[Speaker 2]  We have inflation like very few people have ever seen before, probably the worst in our nation's history. This has been a disaster for people, for the middle class, but for every class.
[Speaker 1]  Donald Trump left us the worst unemployment since the Great Depression. And what we have done is clean up Donald Trump's mess.
[Speaker 2]  She's a Marxist. Everybody knows she's a Marxist. Her father is a Marxist professor in economics, and he taught her well. But her vice presidential pick says abortion in the ninth month is absolutely fine. He also says execution after birth. It's execution no longer abortion because the baby is born is OK. And that's not OK with me.
[Speaker 1]  One does not have to abandon their faith or deeply held beliefs to agree. The government and Donald Trump certainly should not be telling a woman what to do with her body. Pregnant women who want to carry a pregnancy to term suffering from a miscarriage being denied care in an emergency room because the healthcare providers are afraid they might go to jail and she's bleeding out in a car in the parking lot. She didn't want that. Her husband didn't want that.
[Speaker 2]  In Springfield, they're eating the dogs, the people that came in, they're eating the cats, they're eating the pets of the people that live there. And this is what's happening in our country.
[Speaker 3]  the city manager says there's no evidence of that vice president. I'll let you respond to the rest of what you've heard.
[Speaker 2]  Select Einsparter Assence A
[Speaker 1]  It's my day. Alright Alexa, come on there. It is my day. Life.
[Speaker 1]  You talk about extreme.
[Speaker 3]  Are you now acknowledging that you've lost in 2020? No, I don't acknowledge it at all. I said that sarcastically, you know that.
[Speaker 2]  now. I don't acknowledge it at all. I said that sarcastically. You know that. It was said, oh, we lost by a whisker. That was said sarcastically.
[Speaker 3]  Mr. President, thank you. Vice President Harris, you heard the President there tonight. He said he didn't say that, that he lost by Whiskers. So he still believes he did not lose the election. That was won by President Biden and yourself.
[Speaker 1]  Donald Trump was fired by 81 million people. So let's be clear about that. And clearly he is having a very difficult time processing that. World leaders are laughing at Donald Trump. I have talked with military leaders, some of whom work with you. And they say you're a disgrace. Understand what it would mean if Donald Trump were back in the White House with no guardrails. Because certainly we know now the court won't stop him. We know JD Vance is not going to stop him. It's up to the American people.
[Speaker 2]  This is the one that weaponized, not me. She weaponized. I probably took a bullet to their head because of the things that they say about me. They talk about democracy. I'm a threat to democracy. They're the threat to democracy.
[Speaker 1]  So I think you've heard tonight two very different visions for our country, one that is focused on the future, and the other that is focused on the past, and an attempt to take us backward.
[Speaker 1]  but we're not going back.
[Speaker 2]  They've had three and a half years to create jobs and all the things we talked about. Why hasn't she done it? The worst president, the worst vice president in the history of our country.
"""

# Split the transcript into segments for Speakers 1, 2, and 3
def split_transcript_by_speaker(transcript):
    speaker_1_speech = re.findall(r"\[Speaker 1\]([^\[]+)", transcript)
    speaker_2_speech = re.findall(r"\[Speaker 2\]([^\[]+)", transcript)
    speaker_3_speech = re.findall(r"\[Speaker 3\]([^\[]+)", transcript)
    
    speaker_1_text = " ".join(speaker_1_speech).strip()
    speaker_2_text = " ".join(speaker_2_speech).strip()
    speaker_3_text = " ".join(speaker_3_speech).strip()
    
    return speaker_1_text, speaker_2_text, speaker_3_text

# Perform analysis on each speaker's text
speaker_1_text, speaker_2_text, speaker_3_text = split_transcript_by_speaker(transcript)

# Load the sentiment model
sentiment_model, sentiment_tokenizer = load_model(sentiment_model_path)

# Print and Analyze Speaker 1
print("Speaker 1's Text: ")
print(speaker_1_text)  # Print speaker 1 text
print("\nAnalyzing Speaker 1 using RoBERTa...\n")
sentiment_label_1, sentiment_score_1 = analyze_sentiment(speaker_1_text, sentiment_model, sentiment_tokenizer)
ideology_label_1 = analyze_ideology(speaker_1_text, ideology_model_name)

# Print and Analyze Speaker 2
print("\nSpeaker 2's Text: ")
print(speaker_2_text)  # Print speaker 2 text
print("\nAnalyzing Speaker 2 using RoBERTa...\n")
sentiment_label_2, sentiment_score_2 = analyze_sentiment(speaker_2_text, sentiment_model, sentiment_tokenizer)
ideology_label_2 = analyze_ideology(speaker_2_text, ideology_model_name)

# Print and Analyze Speaker 3
print("\nSpeaker 3's Text: ")
print(speaker_3_text)  # Print speaker 3 text
print("\nAnalyzing Speaker 3 using RoBERTa...\n")
sentiment_label_3, sentiment_score_3 = analyze_sentiment(speaker_3_text, sentiment_model, sentiment_tokenizer)
ideology_label_3 = analyze_ideology(speaker_3_text, ideology_model_name)

# Store the results
results = {
    "Model": "RoBERTa",
    "Speaker 1 Sentiment": (sentiment_label_1, sentiment_score_1),
    "Speaker 1 Ideology": ideology_label_1,
    "Speaker 2 Sentiment": (sentiment_label_2, sentiment_score_2),
    "Speaker 2 Ideology": ideology_label_2,
    "Speaker 3 Sentiment": (sentiment_label_3, sentiment_score_3),
    "Speaker 3 Ideology": ideology_label_3
}

df_results = pd.DataFrame([results])
print("\nResults Summary:\n", df_results)


Speaker 1's Text: 
Kamala Harris. What's up, good to be here. You have fun. Thank you.
   Donald Trump left us the worst unemployment since the Great Depression. And what we have done is clean up Donald Trump's mess.
   One does not have to abandon their faith or deeply held beliefs to agree. The government and Donald Trump certainly should not be telling a woman what to do with her body. Pregnant women who want to carry a pregnancy to term suffering from a miscarriage being denied care in an emergency room because the healthcare providers are afraid they might go to jail and she's bleeding out in a car in the parking lot. She didn't want that. Her husband didn't want that.
   It's my day. Alright Alexa, come on there. It is my day. Life.
   You talk about extreme.
   Donald Trump was fired by 81 million people. So let's be clear about that. And clearly he is having a very difficult time processing that. World leaders are laughing at Donald Trump. I have talked with military leaders, s

KeyboardInterrupt: 

## RuBERT

In [14]:
import re
import logging
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
import torch
import pandas as pd

# Disable SSL warnings
import requests
requests.packages.urllib3.disable_warnings(requests.packages.urllib3.exceptions.InsecureRequestWarning)

# Set environment variables
torch.set_num_threads(1)  # Set to 1 thread for optimal performance
logging.getLogger("transformers").setLevel(logging.ERROR)

# Model paths for sentiment and ideology analysis
sentiment_model_path = r"C:\Users\Admin\Downloads\Machine-Learning\Project_part1\Models\rubert-base-cased-sentiment"
ideology_model_name = "EleutherAI/gpt-neo-2.7B"  # Placeholder for ideology model

# Utility function to load models (local)
def load_model(model_path):
    try:
        model = AutoModelForSequenceClassification.from_pretrained(model_path)
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        return model, tokenizer
    except Exception as e:
        logging.error(f"Error loading model from {model_path}: {str(e)}")
        raise e

# Utility to analyze sentiment
def analyze_sentiment(text, model, tokenizer):
    sentiment_analyzer = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)
    
    sentiment_result = sentiment_analyzer(text)
    sentiment_label = sentiment_result[0]["label"]
    sentiment_score = sentiment_result[0]["score"]
    
    # Convert HuggingFace labels to custom labels
    if sentiment_label == 'LABEL_0':  # NEGATIVE
        sentiment_label = 'NEGATIVE'
    elif sentiment_label == 'LABEL_1':  # POSITIVE
        sentiment_label = 'POSITIVE'
    else:  # NEUTRAL
        sentiment_label = 'NEUTRAL'
    
    return sentiment_label, sentiment_score

# Utility to analyze ideology
def analyze_ideology(text, model, tokenizer):
    # Ensure that the tokenizer has a padding token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token  # Set padding token to end-of-sequence (eos_token)

    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, pad_to_multiple_of=8, max_length=512)
    
    input_ids = inputs["input_ids"].long()

    outputs = model(input_ids=input_ids)
    predictions = torch.argmax(outputs.logits, dim=1)

    ideology_labels = ["Progressive", "Conservative", "Neutral"]
    return ideology_labels[predictions.item()]

# Function to split the transcript into speaker blocks
def split_transcript_by_speaker(transcript):
    # Extract all dialog entries based on "[Speaker x]" as the speaker header
    speaker_blocks = re.findall(r"\[Speaker (\d+)\](.*?)\n?(?=\[|$)", transcript, re.DOTALL)
    
    speaker_1_text = ''
    speaker_2_text = ''
    
    for speaker, text in speaker_blocks:
        if speaker == '1':
            speaker_1_text += text.strip() + " "
        elif speaker == '2':
            speaker_2_text += text.strip() + " "
            
    return speaker_1_text.strip(), speaker_2_text.strip()

# Sample transcript
transcript = """
[Speaker 1] Kamala Harris. What's up, good to be here. You have fun. Thank you.
[Speaker 2] We have inflation like very few people have ever seen before, probably the worst in our nation's history. This has been a disaster for people, for the middle class, but for every class.
[Speaker 1] Donald Trump left us the worst unemployment since the Great Depression. And what we have done is clean up Donald Trump's mess.
[Speaker 2] She's a Marxist. Everybody knows she's a Marxist. Her father is a Marxist professor in economics, and he taught her well. But her vice presidential pick says abortion in the ninth month is absolutely fine. He also says execution after birth. It's execution no longer abortion because the baby is born is OK. And that's not OK with me.
[Speaker 1] One does not have to abandon their faith or deeply held beliefs to agree. The government and Donald Trump certainly should not be telling a woman what to do with her body.
[Speaker 2] In Springfield, they're eating the dogs, the people that came in, they're eating the cats, they're eating the pets of the people that live there. And this is what's happening in our country.
[Speaker 2] Now, I don't acknowledge it at all. I said that sarcastically. You know that. It was said, oh, we lost by a whisker. That was said sarcastically.
[Speaker 1] Donald Trump was fired by 81 million people. So let's be clear about that.
[Speaker 2] This is the one that weaponized, not me. She weaponized. I probably took a bullet to their head because of the things that they say about me.
[Speaker 1] So I think you've heard tonight two very different visions for our country.
[Speaker 1] But we're not going back.
[Speaker 2] They've had three and a half years to create jobs and all the things we talked about. Why hasn't she done it?
"""

# Perform analysis on each speaker's text
speaker_1_text, speaker_2_text = split_transcript_by_speaker(transcript)

# Load the sentiment model
sentiment_model, sentiment_tokenizer = load_model(sentiment_model_path)

# Load the ideology model once
ideology_model = AutoModelForSequenceClassification.from_pretrained(ideology_model_name)
ideology_tokenizer = AutoTokenizer.from_pretrained(ideology_model_name)

# Analyze Speaker 1
print("Analyzing Speaker 1 using RuBERT...")
sentiment_label_1, sentiment_score_1 = analyze_sentiment(speaker_1_text, sentiment_model, sentiment_tokenizer)
ideology_label_1 = analyze_ideology(speaker_1_text, ideology_model, ideology_tokenizer)

# Analyze Speaker 2
print("Analyzing Speaker 2 using RuBERT...")
sentiment_label_2, sentiment_score_2 = analyze_sentiment(speaker_2_text, sentiment_model, sentiment_tokenizer)
ideology_label_2 = analyze_ideology(speaker_2_text, ideology_model, ideology_tokenizer)

# Store the results
results = {
    "Model": "RuBERT",
    "Speaker 1 Sentiment": (sentiment_label_1, sentiment_score_1),
    "Speaker 1 Ideology": ideology_label_1,
    "Speaker 2 Sentiment": (sentiment_label_2, sentiment_score_2),
    "Speaker 2 Ideology": ideology_label_2
}

# Display the results
df_results = pd.DataFrame([results])
print(df_results)


Analyzing Speaker 1 using RuBERT...
Analyzing Speaker 2 using RuBERT...
    Model            Speaker 1 Sentiment Speaker 1 Ideology  \
0  RuBERT  (NEUTRAL, 0.7514436841011047)        Progressive   

             Speaker 2 Sentiment Speaker 2 Ideology  
0  (NEUTRAL, 0.7514427304267883)        Progressive  


###### Issues I found while trying different LLM models

1. Warning: Model Weight Initialization Issue
   - Problem:
     When using a general pre-trained model (e.g., sentiment analysis), certain layers (such as the classification layers) are not pre-trained and are initialized randomly. This is causing warnings such as:
     ```
     "Some weights of (model name) were not initialized from the model checkpoint at xxx and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']".
     ```
   - Cause: The layers like the classification head haven't been fine-tuned for a task-specific objective, so they are initialized randomly.
   
   - Solution: Ignore the warning if fine-tuning isn't required, using Python's `warnings` module to suppress it.

2. SSL Connection Errors
   - Problem:
     I am facing several errors related to SSL connections, which prevent downloading models directly from the huggingface.
   - Solution:
     Since I am unable to download models due to SSL issues, I manually downloaded the models and stored them locally. This approach bypasses the SSL-related errors but indicates that the system cannot connect to the remote servers as expected.

## Bibliography

1. Hugging Face. (2024). *DistilBERT base uncased*. Hugging Face Model Hub. Available at: [https://huggingface.co/distilbert-base-uncased](https://huggingface.co/distilbert-base-uncased)

2. Cardiff NLP. (2024). *twitter-roberta-base-sentiment*. Hugging Face Model Hub. Available at: [https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment)

3. EleutherAI. (2024). *GPT-Neo 2.7B*. Hugging Face Model Hub. Available at: [https://huggingface.co/EleutherAI/gpt-neo-2.7B](https://huggingface.co/EleutherAI/gpt-neo-2.7B)

4. Blanchefort, M. (2024). *RuBERT-Tweet Sentiment (blanchefort/rubert-tweet-sentiment-3)*. Hugging Face Model Hub. Available at: [https://huggingface.co/blanchefort/rubert-tweet-sentiment-3](https://huggingface.co/blanchefort/rubert-tweet-sentiment-3)

5. Hugging Face. (2024). *Transformers Documentation*. Available at: [https://huggingface.co/docs/transformers](https://huggingface.co/docs/transformers)

6. Hugging Face. (2024). *Model Hub*. Available at: [https://huggingface.co/models](https://huggingface.co/models)

7. Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A., Kaiser, Ł., Polosukhin, I. (2017). *Attention is All You Need*. NeurIPS 2017. Available at: [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

8. Devlin, J., Chang, M. W., Lee, K., & Toutanova, K. (2019). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL-HLT 2019. Available at: [https://arxiv.org/abs/1810.04805](https://arxiv.org/abs/1810.04805)

9. Sanh, V., Wolf, T., & Ruder, S. (2020). *DistilBERT, a distilled version of BERT: Smaller, faster, cheaper and lighter*. NeurIPS 2020. Available at: [https://arxiv.org/abs/1910.01108](https://arxiv.org/abs/1910.01108)

10. Smith, E. R., & Pustejovsky, J. (2020). *Semantic Processing of Tweets for Sentiment Analysis and Beyond*. Journal of Natural Language Engineering. Available at: [https://doi.org/10.1017/S1351324919000688](https://doi.org/10.1017/S1351324919000688)
